# 020. LSTM/GRU input/output shape

- return_sequences = False, True 일 때의 output 비교

- return_state = False, True 일 때의 internal state output 비교

- Bidirectional LSTM/GRU 의 output 비교

In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional
import numpy as np
import warnings
warnings.filterwarnings('ignore')

B = 2   # 배치 크기 (Batch size): 한 번에 처리할 샘플 개수
T = 5   # 시간 스텝 (Time Steps): 시퀀스의 길이
D = 1   # 특성 개수 (Features): 각 시간 스텝에서의 입력 특성 차원
U = 3   # LSTM 유닛 개수 (LSTM units): LSTM 레이어의 은닉 상태 차원

# 랜덤 입력 데이터 생성
# shape: (배치 크기, 시간 스텝, 특성 개수)
# 예: 2개의 샘플, 각 샘플은 5개의 시간 스텝, 각 스텝은 1개의 특성
X = np.random.randn(B, T, D)

# 입력 데이터의 형태 출력
# 결과: (2, 5, 1) -> 2개 샘플, 5개 시간 스텝, 1개 특성
print(X.shape)

(2, 5, 1)


# LSTM

## return_sequences

- False (default) - last time step 의 output 만 반환
- True - 모든 timestep 의 output 을 모두 반환

<img src="https://i.imgur.com/yqTBCG5.png" width=600 />

In [2]:
def lstm(return_sequences=False):
    """
    Parameters:
    - return_sequences: False면 마지막 시간 스텝의 출력만 반환,
                       True면 모든 시간 스텝의 출력을 반환
    """
    # 입력 레이어 정의: (시간 스텝, 특성 개수) 형태
    inp = Input(shape=(T, D))

    # LSTM 레이어 적용
    # U: LSTM 유닛 개수 (은닉 상태의 차원)
    # return_sequences: 모든 시간 스텝의 출력 반환 여부
    out = LSTM(U, return_sequences=return_sequences)(inp)

    # 모델 생성 및 예측 수행
    model = Model(inputs=inp, outputs=out)
    return model.predict(X)

print("---- return_sequences=False ----> 마지막 시간 스텝의 출력만 반환")
# 마지막 시간 스텝(t=5)의 은닉 상태만 출력
# 결과 shape: (B, U) = (2, 3)
lstm_out = lstm(return_sequences=False)
print(lstm_out.shape)
print(lstm_out)

print("\n---- return_sequences=True ----> 모든 시간 스텝별 출력 반환")
# 모든 시간 스텝(t=1,2,3,4,5)의 은닉 상태를 출력
# 결과 shape: (B, T, U) = (2, 5, 3)
lstm_out = lstm(return_sequences=True)
print(lstm_out.shape)
print(lstm_out)

---- return_sequences=False ----> 마지막 시간 스텝의 출력만 반환
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step
(2, 3)
[[ 0.12372325  0.01077074 -0.00314797]
 [ 0.02754275 -0.06651854  0.03978407]]

---- return_sequences=True ----> 모든 시간 스텝별 출력 반환
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
(2, 5, 3)
[[[ 0.04345136  0.03164463  0.10007914]
  [-0.00612845  0.01506977  0.04369809]
  [ 0.0456823   0.04163887  0.13212386]
  [-0.18452692 -0.2798394  -0.09483339]
  [-0.13281602 -0.09563949  0.00669932]]

 [[ 0.06205368  0.02916952  0.11837366]
  [ 0.04898045  0.06173255  0.17171514]
  [ 0.04034115  0.07032439  0.15848057]
  [ 0.03779415  0.07773463  0.15447587]
  [-0.05481488  0.01290678  0.02064373]]]


## return_state

- False (default) - output 만 반환

- True - output, last step 의 hidden state, cell state (LSTM 의 경우) 반환

In [3]:
def lstm(return_state=False):
    """
    Parameters:
    - return_state: False면 출력만 반환,
                   True면 출력, 은닉 상태, 셀 상태를 모두 반환
    """
    # 입력 레이어 정의: (시간 스텝, 특성 개수) 형태
    inp = Input(shape=(T, D))

    # LSTM 레이어 적용
    # return_state=True: 마지막 은닉 상태(h)와 셀 상태(c)도 함께 반환
    out = LSTM(U, return_state=return_state)(inp)

    # 모델 생성 및 예측
    model = Model(inputs=inp, outputs=out)

    if return_state:
        # return_state=True인 경우: (출력, 은닉 상태, 셀 상태) 반환
        o, h, c = model.predict(X)
        print("o (output) :", o.shape)  # 출력: 마지막 시간 스텝의 출력
        print(o)
        print("h (hidden state) :", h.shape)  # 은닉 상태: 단기 메모리
        print(h)
        print("c (cell state) :", c.shape)  # 셀 상태: 장기 메모리
        print(c)
    else:
        # return_state=False인 경우: 출력만 반환
        o = model.predict(X)
        print("o (output) :", o.shape)
        print(o)

print("---- return_state=False ----> 출력(output)만 반환")
lstm(return_state=False)

print("\n---- return_state=True ----> 출력(output), 은닉 상태(hidden state), 셀 상태(cell state) 모두 반환")
lstm(return_state=True)

---- return_state=False ----> 출력(output)만 반환
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
o (output) : (2, 3)
[[ 0.00033575 -0.00465709  0.04761355]
 [-0.01826128 -0.03866781 -0.00066639]]

---- return_state=True ----> 출력(output), 은닉 상태(hidden state), 셀 상태(cell state) 모두 반환
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
o (output) : (2, 3)
[[ 0.0375324  -0.07614231 -0.02767035]
 [ 0.04942595  0.020559    0.03789097]]
h (hidden state) : (2, 3)
[[ 0.0375324  -0.07614231 -0.02767035]
 [ 0.04942595  0.020559    0.03789097]]
c (cell state) : (2, 3)
[[ 0.07524515 -0.14386782 -0.05254363]
 [ 0.0918295   0.04701845  0.08797702]]


# Bidirectional LSTM

- 순방향, 역방향이 concatenate 된 output 출력  

- hidden state, cell state 는 순방향, 역방향 별도 출력

In [4]:
T, D, U

(5, 1, 3)

In [5]:
def bi_lstm(return_sequences=False, return_state=False):
    """
    Parameters:
    - return_sequences: False면 마지막 시간 스텝의 출력만 반환,
                       True면 모든 시간 스텝의 출력을 반환
    - return_state: False면 출력만 반환,
                   True면 출력과 함께 순방향/역방향의 은닉 상태, 셀 상태를 반환
    """
    # 입력 레이어 정의: (시간 스텝, 특성 개수) 형태
    inp = Input(shape=(T, D))

    # 양방향 LSTM 레이어 적용
    # Bidirectional: 순방향(forward)과 역방향(backward) 두 방향으로 LSTM 실행
    # 순방향: 시간 순서대로 처리 (t=1 → t=5)
    # 역방향: 시간 역순으로 처리 (t=5 → t=1)
    out = Bidirectional(
            LSTM(U, return_state=return_state, return_sequences=return_sequences))(inp)

    # 모델 생성 및 예측
    model = Model(inputs=inp, outputs=out)

    if return_state:
        # return_state=True인 경우: 출력 + 순방향/역방향 상태 반환
        # o: 출력 (순방향과 역방향이 연결됨)
        # h1, c1: 순방향 LSTM의 은닉 상태와 셀 상태
        # h2, c2: 역방향 LSTM의 은닉 상태와 셀 상태
        o, h1, c1, h2, c2 = model.predict(X)
        print("o (output - forward + backward 연결) :", o.shape)
        print("h1 (순방향 hidden state) :", h1.shape)
        print("c1 (순방향 cell state) :", c1.shape)
        print("h2 (역방향 hidden state) :", h2.shape)
        print("c2 (역방향 cell state) :", c2.shape)
    else:
        # return_state=False인 경우: 출력만 반환
        o = model.predict(X)
        print("o (output - forward + backward 연결) :", o.shape)

print("*** 순방향, 역방향 출력이 concatenate되어 차원이 2배 ***")
print("---- return_sequences=False ----> 마지막 시간 스텝의 출력만 반환")
# 출력 shape: (B, U*2) = (2, 6)
# 순방향 U개 + 역방향 U개 = 총 2*U개
bi_lstm(return_sequences=False, return_state=False)

print()
print("---- return_sequences=True ----> 모든 시간 스텝별 출력 반환")
# 출력 shape: (B, T, U*2) = (2, 5, 6)
# 각 시간 스텝마다 순방향 + 역방향 출력이 연결됨
bi_lstm(return_sequences=True)

print()
print("---- return_sequences=True, return_state=True")

*** 순방향, 역방향 출력이 concatenate되어 차원이 2배 ***
---- return_sequences=False ----> 마지막 시간 스텝의 출력만 반환


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 295ms/step
o (output - forward + backward 연결) : (2, 6)

---- return_sequences=True ----> 모든 시간 스텝별 출력 반환


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step
o (output - forward + backward 연결) : (2, 5, 6)

---- return_sequences=True, return_state=True
